# Operator actions and Symbolica export

This notebook walks through the **compiled Lagrangian** layer:

- **`Model` → `model.lagrangian()`** gives a `CompiledLagrangian` of ordered `InteractionTerm`s (the structure the vertex code trusts).
- **`L.to_symbolica()`** turns that into a single Symbolica expression for **display / scalar algebra** (factor order for fermions is **not** preserved in Symbolica).
- **`L.apply_operator(...)`** applies a graded Leibniz rule at the term level; optional **`flavor_expand`** matches `feynman_rule`.

Operators are configured with **`lagrangian.operator_action`** (small helpers like `replacement_operator`); everything else below uses the **model API** on the compiled object.

In [26]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (ROOT, SRC):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from symbolica import Expression, S

from model import Model, PartialD, dirac_field, flavor_index, scalar_field
from lagrangian.operator_action import (
    FieldOperator,
    partial,
    replacement_operator,
    single_field_result,
)

## 1. Define fields and build the model

In [27]:
def field_factor(occ):
    name = occ.field.name
    if occ.conjugated and not occ.field.self_conjugate:
        name = name + ".bar"
    return name


def describe_term(t, index):
    factors = " · ".join(field_factor(o) for o in t.fields)
    if t.derivatives:
        d = ", ".join(f"slot {a.target} → ∂_{a.lorentz_index}" for a in t.derivatives)
    else:
        d = "(none)"
    bil = t.closed_dirac_bilinears or "(none)"
    print(f"[{index}] {factors}")
    print(f"     ∂: {d}   bilinears: {bil}")



In [28]:
Generation = flavor_index("Generation", 3, prefix="f")
mu = S("mu")
lam = S("lam")
f = S("f")


Chi = scalar_field("Chi", self_conjugate=True)
Phi = scalar_field("Phi", self_conjugate=True)
psi = dirac_field("psi", indices=())
eta = dirac_field("eta", indices=())

lepton = dirac_field(
    "l",
    class_members=("e", "mu", "ta"),
    indices=(Generation,),
    flavor_index=Generation,
)
H = scalar_field("H", self_conjugate=True)



In [29]:

model_phi3 = Model(Phi * Phi * Phi + Phi * PartialD(Phi, mu))
L_phi3 = model_phi3.lagrangian()

## 3. Symbolica export (`to_symbolica`)


In [30]:
expr = L_phi3.to_symbolica()
print(expr)
print("Simplified:", Expression.parse(str(expr)).expand())

Phi*PartialD(Phi,mu)+Phi^3
Simplified: Phi*PartialD(Phi,mu)+Phi^3


## 4. Apply a field operator (`apply_operator`)

**Simple replacement:** map every `Phi` factor to `Chi` 

In [31]:
replace_phi = replacement_operator("Phi_to_Chi", {Phi: Chi()})
L_replaced = L_phi3.apply_operator(replace_phi)

print("After replacing each Phi by Chi:", len(L_replaced.terms), "term(s)")
for i, term in enumerate(L_replaced.terms):
    describe_term(term, i)

print("\nSymbolica view:")
print(L_replaced.to_symbolica())
print("feynman rules view:", L_replaced.feynman_rule())

After replacing each Phi by Chi: 5 term(s)
[0] Chi · Phi · Phi
     ∂: (none)   bilinears: (none)
[1] Phi · Chi · Phi
     ∂: (none)   bilinears: (none)
[2] Phi · Phi · Chi
     ∂: (none)   bilinears: (none)
[3] Chi · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[4] Phi · Chi
     ∂: slot 1 → ∂_mu   bilinears: (none)

Symbolica view:
Chi*PartialD(Phi,mu)+3*Chi*Phi^2+Phi*PartialD(Chi,mu)
feynman rules view: {('Chi', 'Phi', 'Phi'): 6𝑖, ('Chi', 'Phi'): pcomp(q1,mu1_int)+pcomp(q2,mu1_int)}


### Phi → Chi Phi

In [32]:
split_phi = replacement_operator(
    "split_phi",
    {Phi: single_field_result((Chi(), Phi()))},
)
L_split = L_phi3.apply_operator(split_phi)
print("O[L_phi3]=", L_split.to_symbolica())
print("=====================================================================")
print("Product replacement on ∂ slots →", len(L_split.terms), "terms (derivative Leibniz).")
for i, term in enumerate(L_split.terms):
    describe_term(term, i)
print("=====================================================================")
print("Feynman rules view:", L_split.feynman_rule())

O[L_phi3]= 2*Chi*Phi*PartialD(Phi,mu)+3*Chi*Phi^3+Phi^2*PartialD(Chi,mu)
Product replacement on ∂ slots → 6 terms (derivative Leibniz).
[0] Chi · Phi · Phi · Phi
     ∂: (none)   bilinears: (none)
[1] Phi · Chi · Phi · Phi
     ∂: (none)   bilinears: (none)
[2] Phi · Phi · Chi · Phi
     ∂: (none)   bilinears: (none)
[3] Chi · Phi · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)
[4] Phi · Chi · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[5] Phi · Chi · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)
Feynman rules view: {('Chi', 'Phi', 'Phi', 'Phi'): 18𝑖, ('Chi', 'Phi', 'Phi'): 2*pcomp(q1,mu1_int)+2*pcomp(q2,mu1_int)+2*pcomp(q3,mu1_int)}


## 5. Flavor-expanded export (`flavor_expand`)

Same keyword as for vertices: pass **`flavor_expand=True`** (or specific flavor indices) so **`to_symbolica`** serializes the **expanded** interaction list (class members like `e`, `mu`, `ta` instead of the generic `l`).

You can pass the same flag to **`apply_operator(..., flavor_expand=True)`** if you want the operator to act on that expanded view first.

In [33]:
flavor_model = Model(S("g") * lepton.bar(f) * lepton(f) * H)
L_fl = flavor_model.lagrangian()

compact = L_fl.to_symbolica()
expanded = L_fl.to_symbolica(flavor_expand=True)

print("Generic (class field 'l'):")
print(compact)
print()
print("Flavor-expanded (e, mu, ta):")
print(expanded)

Generic (class field 'l'):
H*g*l(f,i_decl_1)*lbar(f,i_decl_1)

Flavor-expanded (e, mu, ta):
H*g*mu(i_decl_1)*mubar(i_decl_1)+H*g*ebar(i_decl_1)*e(i_decl_1)+H*g*tabar(i_decl_1)*ta(i_decl_1)


## 6. Odd operator sign (fermion bookkeeping)

In [34]:
fermion_model = Model(
    psi.bar() * psi()*psi.bar() * psi(),
)
L_f = fermion_model.lagrangian()


def odd_s_on_psi(occurrence):
    if occurrence.field is not psi:
        return None
    return single_field_result(eta.occurrence(conjugated=occurrence.conjugated))


s_odd = FieldOperator(name="s", parity=1, on_field=odd_s_on_psi)
L_s = L_f.apply_operator(s_odd)

print("Original:", len(L_f.terms), "term(s)")
for t in L_f.terms:
    print("  coupling", t.coupling, "→", " · ".join(field_factor(o) for o in t.fields))

print("\nAfter odd s on each psi slot:", len(L_s.terms), "terms")
print("O[L_f]=", L_s.to_symbolica())
for t in L_s.terms:
    print("  coupling", t.coupling, "→", " · ".join(field_factor(o) for o in t.fields))

Original: 1 term(s)
  coupling 1 → psi.bar · psi · psi.bar · psi

After odd s on each psi slot: 4 terms
O[L_f]= psi(i_decl_1)*psi(i_decl_2)*psibar(i_decl_1)*etabar(i_decl_2)+psi(i_decl_1)*psi(i_decl_2)*psibar(i_decl_2)*etabar(i_decl_1)-psi(i_decl_1)*psibar(i_decl_1)*psibar(i_decl_2)*eta(i_decl_2)-psi(i_decl_2)*psibar(i_decl_1)*psibar(i_decl_2)*eta(i_decl_1)
  coupling 1 → eta.bar · psi · psi.bar · psi
  coupling -1 → psi.bar · eta · psi.bar · psi
  coupling 1 → psi.bar · psi · eta.bar · psi
  coupling -1 → psi.bar · psi · psi.bar · eta


## 7. Scalar total-derivative investigation

Goal: investigate the minimal scalar identity

$$\partial_\mu(\phi^2\,\partial_\mu\phi)
= 2\,\phi\,(\partial_\mu\phi)(\partial_\mu\phi) + \phi^2\,\Box\phi$$

and therefore, **only modulo a total derivative / boundary term under the spacetime integral**, 

$$\phi^2\,\Box\phi \simeq -2\,\phi\,(\partial_\mu\phi)(\partial_\mu\phi).$$

In [35]:
c = S("c")
phi = Phi

current_model = Model(
    phi * phi * PartialD(phi, mu),
)

expanded_divergence_model = Model(
    (
        2 * phi * PartialD(phi, mu) * PartialD(phi, mu)
        + phi * phi * PartialD(PartialD(phi, mu), mu)
    ),
)

L1_model = Model(
    c * phi * phi * PartialD(PartialD(phi, mu), mu),
)

L2_model = Model(
    -2 * c * phi * PartialD(phi, mu) * PartialD(phi, mu),
)

delta_model = Model(
    (
        c * phi * phi * PartialD(PartialD(phi, mu), mu)
        + 2 * c * phi * PartialD(phi, mu) * PartialD(phi, mu)
    ),
)

J = current_model.lagrangian()
J_expanded = expanded_divergence_model.lagrangian()
L1_td = L1_model.lagrangian()
L2_td = L2_model.lagrangian()
Delta_td = delta_model.lagrangian()

### 7.1 Current and present operator-layer status

The public DSL can represent the current $J^\mu = \phi^2\,\partial_\mu\phi$ directly.

What we **cannot** currently do naturally is ask the present `FieldOperator` replacement layer to generate the **outer** $\partial_\mu(\ldots)$ from scratch, because it replaces field slots and redistributes derivatives that already exist on slots; it does not create a fresh `DerivativeAction` on an otherwise underived product.

In [36]:
print("Current source:", current_model.lagrangian_decl.source_terms[0])
print()
for i, term in enumerate(J.terms):
    describe_term(term, i)

assert len(J.terms) == 1
assert [occ.field.name for occ in J.terms[0].fields] == ["Phi", "Phi", "Phi"]
assert [(a.target, str(a.lorentz_index)) for a in J.terms[0].derivatives] == [(2, "mu")]

print("Symbolica export:")
print(J.to_symbolica())

Current source: Phi * Phi * PartialD(Phi, mu)

[0] Phi · Phi · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)
Symbolica export:
Phi^2*PartialD(Phi,mu)


### 7.2 Manual expansion of the divergence

Since the current operator-action layer does not yet create the fresh outer derivative, we build the right-hand side directly with the existing public `PartialD(...)` API and inspect the lowered terms.

In [37]:
print("Expanded source:", expanded_divergence_model.lagrangian_decl.source_terms[0])
print()
for i, term in enumerate(J_expanded.terms):
    describe_term(term, i)

assert len(J_expanded.terms) == 2
assert J_expanded.terms[0].coupling == 2
assert J_expanded.terms[1].coupling == 1
assert [(a.target, str(a.lorentz_index)) for a in J_expanded.terms[0].derivatives] == [(1, "mu"), (2, "mu")]
assert [(a.target, str(a.lorentz_index)) for a in J_expanded.terms[1].derivatives] == [(2, "mu"), (2, "mu")]

print("Symbolica export:")
print(J_expanded.to_symbolica())
print("Expanded:")
print(Expression.parse(str(J_expanded.to_symbolica())).expand())

Expanded source: 2 * Phi * PartialD(Phi, mu) * PartialD(Phi, mu)

[0] Phi · Phi · Phi
     ∂: slot 1 → ∂_mu, slot 2 → ∂_mu   bilinears: (none)
[1] Phi · Phi · Phi
     ∂: slot 2 → ∂_mu, slot 2 → ∂_mu   bilinears: (none)
Symbolica export:
2*Phi*PartialD(Phi,mu)^2+Phi^2*PartialD(PartialD(Phi,mu),mu)
Expanded:
2*Phi*PartialD(Phi,mu)^2+Phi^2*PartialD(PartialD(Phi,mu),mu)


### 7.3 Compare $L_1$, $L_2$, and $L_1 - L_2$

Define

$$L_1 = c\,\phi^2\,\Box\phi, \qquad L_2 = -2c\,\phi\,(\partial_\mu\phi)(\partial_\mu\phi).$$

Then

$$L_1 - L_2 = c\,\partial_\mu(\phi^2\,\partial_\mu\phi).$$

So the two Lagrangians are equivalent **modulo a total derivative**, not as identical local expressions.

In [38]:
expr_J_expanded = J_expanded.to_symbolica()
expr_L1 = L1_td.to_symbolica()
expr_L2 = L2_td.to_symbolica()
expr_Delta = Delta_td.to_symbolica()
expr_cJ = c * expr_J_expanded
expr_delta_check = (expr_Delta - expr_cJ).expand()

print("L1 =")
print(expr_L1)
print()
print("L2 =")
print(expr_L2)
print()
print("L1 - L2 =")
print(expr_Delta)
print()
print("c * expanded divergence =")
print(expr_cJ)
print()
print("Expanded check:")
print(expr_delta_check)

assert expr_delta_check.to_canonical_string() == "0"
print()
print("Check passed: L1 - L2 matches c * ∂_μ(φ² ∂_μ φ) after manual expansion.")


L1 =
Phi^2*c*PartialD(PartialD(Phi,mu),mu)

L2 =
-2*Phi*c*PartialD(Phi,mu)^2

L1 - L2 =
2*Phi*c*PartialD(Phi,mu)^2+Phi^2*c*PartialD(PartialD(Phi,mu),mu)

c * expanded divergence =
c*(2*Phi*PartialD(Phi,mu)^2+Phi^2*PartialD(PartialD(Phi,mu),mu))

Expanded check:
0

Check passed: L1 - L2 matches c * ∂_μ(φ² ∂_μ φ) after manual expansion.


### 7.4 Conclusion

- The framework can represent both sides of this scalar identity with the current public declarative API.
- A real spacetime derivative as a runtime operator now exists: `partial(mu)` — see section 8 — and reproduces, slot by slot, the same lowered structure as the declarative `PartialD(...)`.
- `to_symbolica()` is useful here for scalar display and algebra checks **after** lowering.
- Symbolica does **not** know the physics statement that a total derivative only contributes a boundary term under $\int d^4x$.
- A future automatic IBP / total-derivative simplification layer should sit above the lowered `InteractionTerm` representation, and should stay separate from `vertex_engine.py` and separate from the `partial(mu)` operator itself.

## 8. Spacetime derivative operator: `partial(mu)`

In [39]:
cubic = Model(lam * Phi * Phi * Phi).lagrangian()

L_partial = cubic.apply_operator(partial(mu))

print("Source:           lam * Phi * Phi * Phi")
print("partial(mu)[L]:  ", L_partial.to_symbolica())
print()
for i, term in enumerate(L_partial.terms):
    describe_term(term, i)

print("======================================================================")
print("Feynman rules view:", L_partial.feynman_rule(Phi,Phi,Phi, include_delta=True))

Source:           lam * Phi * Phi * Phi
partial(mu)[L]:   3*lam*Phi^2*PartialD(Phi,mu)

[0] Phi · Phi · Phi
     ∂: slot 0 → ∂_mu   bilinears: (none)
[1] Phi · Phi · Phi
     ∂: slot 1 → ∂_mu   bilinears: (none)
[2] Phi · Phi · Phi
     ∂: slot 2 → ∂_mu   bilinears: (none)
Feynman rules view: 6*lam*(2*𝜋)^d*pcomp(q1,mu1_int)*Delta(q1+q2+q3)+6*lam*(2*𝜋)^d*pcomp(q2,mu1_int)*Delta(q1+q2+q3)+6*lam*(2*𝜋)^d*pcomp(q3,mu1_int)*Delta(q1+q2+q3)


### 8.1 partial(mu) on Phi * Chi

In [40]:
product_model = Model(Phi * Chi).lagrangian()
print("partial(mu)[Phi * Chi] =", product_model.apply_operator(partial(mu)).to_symbolica())

restricted = product_model.apply_operator(partial(mu, on=Phi))
print("partial(mu, on=Phi)[Phi * Chi] =", restricted.to_symbolica())


partial(mu)[Phi * Chi] = Chi*PartialD(Phi,mu)+Phi*PartialD(Chi,mu)
partial(mu, on=Phi)[Phi * Chi] = Chi*PartialD(Phi,mu)


## 9. SU(3) × SU(2) × U(1) export check

In [41]:
from model import GaugeFixing, build_unbroken_standard_model

sm = build_unbroken_standard_model()
xiB = S("xiB")
xiW = S("xiW")
xiG = S("xiG")

sm_decl_with_gf = (
    sm.lagrangians.LSM
    + GaugeFixing(sm.gauge_groups.U1Y, xi=xiB)
    + GaugeFixing(sm.gauge_groups.SU2L, xi=xiW)
    + GaugeFixing(sm.gauge_groups.SU3C, xi=xiG)
)

sm_model_with_gf = Model(
    lagrangian_decl= sm_decl_with_gf,
    name="SM-unbroken-with-gf",
    gauge_groups=(sm.gauge_groups.U1Y, sm.gauge_groups.SU2L, sm.gauge_groups.SU3C),
    parameters=sm.model.parameters,
)
L_sm_gf = sm_model_with_gf.lagrangian()
expr_sm_gf = L_sm_gf.to_symbolica()

print("Full SM with gauge fixing, Symbolica export:",expr_sm_gf)
expr_sm_gf_text = str(expr_sm_gf)

print("Gauge groups:", [group.name for group in sm_model_with_gf.gauge_groups])
print("Source declarations:", len(sm_decl_with_gf.source_terms))
print("Lowered terms:", len(L_sm_gf.terms))
print("Symbolica string length:", len(expr_sm_gf_text))
print()
print("Preview:")
print(expr_sm_gf_text[:1500] + " ...")

assert "PartialD(" in expr_sm_gf_text
assert "CovD(" not in expr_sm_gf_text
assert "FieldStrength(" not in expr_sm_gf_text
assert "GaugeFixing(" not in expr_sm_gf_text
assert "xiB" in expr_sm_gf_text and "xiW" in expr_sm_gf_text and "xiG" in expr_sm_gf_text
print()
print("Check passed: the full SU(3)xSU(2)xU(1) Lagrangian with gauge fixing exports to lowered Symbolica form.")


Full SM with gauge fixing, Symbolica export: -lam*Phi0(w_decl_1)*Phi0(w_decl_2)*Phibar0(w_decl_1)*Phibar0(w_decl_2)+1/2*g1*g2*g(mink(4, mu),mink(4, nu))*t(coad(3, weak_adj_Wi_SU2L_mix),cof(2, w_bar_Phi),cof(2, w_Phi))*Phi0(w_Phi)*Phibar0(w_bar_Phi)*B0(mu)*Wi0(nu,weak_adj_Wi_SU2L_mix)+1/2*g1*g2*g(mink(4, mu),mink(4, nu))*t(coad(3, weak_adj_Wi_SU2L_mix),cof(2, w_bar_Phi),cof(2, w_Phi))*Phi0(w_Phi)*Phibar0(w_bar_Phi)*B0(nu)*Wi0(mu,weak_adj_Wi_SU2L_mix)+1/6*g1*g(cof(2, w_bar_qL_weak_fund_id),cof(2, w_qL_weak_fund_id))*g(cof(3, fl_bar_qL_generation_id),cof(3, fl_qL_generation_id))*g(cof(3, c_bar_qL_color_fund_id),cof(3, c_qL_color_fund_id))*gamma(bis(4, i_bar_qL_U1Y),bis(4, i_qL_U1Y),mink(4, mu))*qL0(i_qL_U1Y,w_qL_weak_fund_id,fl_qL_generation_id,c_qL_color_fund_id)*qLbar0(i_bar_qL_U1Y,w_bar_qL_weak_fund_id,fl_bar_qL_generation_id,c_bar_qL_color_fund_id)*B0(mu)-1/2*g1*g(cof(2, w_bar_lL_weak_fund_id),cof(2, w_lL_weak_fund_id))*g(cof(3, fl_bar_lL_generation_id),cof(3, fl_lL_generation_id))*ga

## 10. Replacement-style operator checks



In [42]:
from lagrangian.operator_action import OperatorAtomResult, OperatorSummand

g = S("g")
c = S("c")


### 10.1 Odd-operator signs

Test the graded Leibniz signs on $L = g\,\bar\psi\,\psi\,\phi$ with an odd operator $s$.


In [43]:
psi = dirac_field("psi", indices=())
xi = dirac_field("xi", indices=())
phi = scalar_field("phi", self_conjugate=True)
chi= scalar_field("chi", self_conjugate=True)

odd_model = Model(
    g * psi.bar() * psi() * phi,
)
L_odd_slots = odd_model.lagrangian()


def s_on_field_slot(occ):
    if occ.field is psi:
        return single_field_result((xi(conjugated=occ.conjugated),))
    if occ.field is phi:
        return single_field_result((chi(),))
    return None


s_slot = FieldOperator(name="s", parity=1, on_field=s_on_field_slot)
L_odd_out = L_odd_slots.apply_operator(s_slot)

print("Source:", odd_model.lagrangian_decl.source_terms[0])
print()
for i, term in enumerate(L_odd_out.terms):
    print("coupling", term.coupling)
    describe_term(term, i)
print()
print("symbolica preview:", L_odd_slots.to_symbolica())
print("Symbolica export:")
print(L_odd_out.to_symbolica())

assert [term.coupling for term in L_odd_out.terms] == [g, -g, g]
assert [[field_factor(o) for o in term.fields] for term in L_odd_out.terms] == [
    ["xi.bar", "psi", "phi"],
    ["psi.bar", "xi", "phi"],
    ["psi.bar", "psi", "chi"],
]
print()
print("Feynman rules for (xi.bar, psi, phi):", L_odd_out.feynman_rule(xi.bar,psi,phi))
print("Feynman rules for (psi.bar, xi, phi):", L_odd_out.feynman_rule(psi.bar,xi,phi))
print("Feynman rules for (psi.bar, psi, chi):", L_odd_out.feynman_rule(psi.bar,psi,chi))


Source: g * psi.bar * psi * phi

coupling g
[0] xi.bar · psi · phi
     ∂: (none)   bilinears: ((0, 1),)
coupling -g
[1] psi.bar · xi · phi
     ∂: (none)   bilinears: ((0, 1),)
coupling g
[2] psi.bar · psi · chi
     ∂: (none)   bilinears: ((0, 1),)

symbolica preview: phi*g*psi(i_decl_1)*psibar(i_decl_1)
Symbolica export:
phi*g*psi(i_decl_1)*xibar(i_decl_1)-phi*g*psibar(i_decl_1)*xi(i_decl_1)+g*chi*psi(i_decl_1)*psibar(i_decl_1)

Feynman rules for (xi.bar, psi, phi): 1𝑖*g*g(bis(4, i1),bis(4, i2))
Feynman rules for (psi.bar, xi, phi): -1𝑖*g*g(bis(4, i1),bis(4, i2))
Feynman rules for (psi.bar, psi, chi): 1𝑖*g*g(bis(4, i1),bis(4, i2))


### 10.2 Multiple summands per slot

One field can vary into a sum, so the slot contribution itself may have several summands.


In [44]:
phi_sum = scalar_field("phi_sum", self_conjugate=True)
eta_sum = scalar_field("eta_sum", self_conjugate=True)
A_sum = scalar_field("A_sum", self_conjugate=True)
B_sum = scalar_field("B_sum", self_conjugate=True)
R_sum = scalar_field("R_sum", self_conjugate=True)

sum_model = Model(
    g * phi_sum * eta_sum,
)
L_sum_slots = sum_model.lagrangian()


def O_sum_slot(occ):
    if occ.field is phi_sum:
        return OperatorAtomResult(
            summands=(
                OperatorSummand(replacement=(A_sum(),)),
                OperatorSummand(replacement=(B_sum(),)),
            )
        )
    if occ.field is eta_sum:
        return single_field_result((R_sum(),))
    return None


O_sum = FieldOperator(name="O_sum", parity=0, on_field=O_sum_slot)
L_sum_out = L_sum_slots.apply_operator(O_sum)
expr_sum_out = L_sum_out.to_symbolica()

print("Source:", sum_model.lagrangian_decl.source_terms[0])
print()
for i, term in enumerate(L_sum_out.terms):
    print("coupling", term.coupling)
    describe_term(term, i)
print()
print("Symbolica export:")
print(expr_sum_out)

assert len(L_sum_out.terms) == 3
assert [[field_factor(o) for o in term.fields] for term in L_sum_out.terms] == [
    ["A_sum", "eta_sum"],
    ["B_sum", "eta_sum"],
    ["phi_sum", "R_sum"],
]
assert expr_sum_out.expand().to_canonical_string() == Expression.parse(
    "g*A_sum*eta_sum + g*B_sum*eta_sum + g*phi_sum*R_sum"
).to_canonical_string()


Source: g * phi_sum * eta_sum

coupling g
[0] A_sum · eta_sum
     ∂: (none)   bilinears: (none)
coupling g
[1] B_sum · eta_sum
     ∂: (none)   bilinears: (none)
coupling g
[2] phi_sum · R_sum
     ∂: (none)   bilinears: (none)

Symbolica export:
g*phi_sum*R_sum+g*eta_sum*A_sum+g*eta_sum*B_sum


### 10.3 Derivative fan-out

If the operator commutes with partial derivatives, an existing derivative on a slot fans out across a product-valued replacement.


In [45]:
phi_df = scalar_field("phi_df", self_conjugate=True)
chi_df = scalar_field("chi_df", self_conjugate=True)
eta_df = scalar_field("eta_df", self_conjugate=True)

fan_model = Model(c * PartialD(phi_df, mu),
)
L_fan_slots = fan_model.lagrangian()
O_fan = replacement_operator(
    "O_fan",
    {phi_df: single_field_result((chi_df(), eta_df()))},
)
L_fan_out = L_fan_slots.apply_operator(O_fan)
expr_fan_out = L_fan_out.to_symbolica()

print("Source:", fan_model.lagrangian_decl.source_terms[0])
print()
for i, term in enumerate(L_fan_out.terms):
    print("coupling", term.coupling)
    describe_term(term, i)
print()
print("Symbolica export:")
print(expr_fan_out)

assert len(L_fan_out.terms) == 2
assert [[field_factor(o) for o in term.fields] for term in L_fan_out.terms] == [
    ["chi_df", "eta_df"],
    ["chi_df", "eta_df"],
]
assert [[(d.target, str(d.lorentz_index)) for d in term.derivatives] for term in L_fan_out.terms] == [
    [(0, "mu")],
    [(1, "mu")],
]
assert expr_fan_out.expand().to_canonical_string() == Expression.parse(
    "c*(PartialD(chi_df,mu)*eta_df + chi_df*PartialD(eta_df,mu))"
).expand().to_canonical_string()
print()
print("Feynman rules for (chi_df, eta_df):", L_fan_out.feynman_rule(chi_df, eta_df))

Source: c * PartialD(phi_df, mu)

coupling c
[0] chi_df · eta_df
     ∂: slot 0 → ∂_mu   bilinears: (none)
coupling c
[1] chi_df · eta_df
     ∂: slot 1 → ∂_mu   bilinears: (none)

Symbolica export:
c*chi_df*PartialD(eta_df,mu)+c*eta_df*PartialD(chi_df,mu)

Feynman rules for (chi_df, eta_df): c*pcomp(q1,mu1_int)+c*pcomp(q2,mu1_int)


## 11. Infinitesimal gauge variation: `gauge_variation`

The operator layer above is just *replacement plus graded Leibniz on lowered slots*. **Gauge transformations** are exactly this in disguise. For one gauge group, the **infinitesimal** variation acts as

* matter rep R: $\delta \Phi_i = + i g\, \alpha^a T^a_{ij}\, \Phi_j$, $\delta \bar\Phi_i = - i g\, \alpha^a \bar\Phi_j T^a_{ji}$
* abelian U(1) charge $q$: $\delta \Phi = + i g q\, \alpha\, \Phi$, $\delta \bar\Phi = - i g q\, \alpha\, \bar\Phi$
* gauge boson: $\delta A^a_\mu = +\partial_\mu \alpha^a - g f^{abc} \alpha^b A^c_\mu$ (non-abelian); $\delta A_\mu = +\partial_\mu \alpha$ (abelian)

 `D_μ = ∂_μ - i g A_μ` convention (see `compiler/gauge.py`).

`gauge_variation(group=..., parameter="alpha")` returns a `FieldOperator` we can hand to `L.apply_operator(...)`. Under the hood, $\alpha$ is materialised as a synthetic scalar field with the group's adjoint index (so existing derivatives on the acted slot Leibniz-distribute across $(\alpha, \Phi)$ correctly — no new engine machinery needed).

In [46]:
from model import CovD, Gamma, GaugeGroup, Field, LORENTZ_INDEX
from lagrangian.operator_action import gauge_variation
from symbolic.tensor_canonicalization import canonize_full

# A tiny U(1) example: charged complex scalar + charged Dirac fermion.
g_u1 = S("g")
A_u1 = Field("A", spin=1, indices=(LORENTZ_INDEX,), self_conjugate=True)
U1 = GaugeGroup(name="U1", abelian=True, coupling=g_u1, charge="Q", gauge_boson=A_u1)

Phi_q = scalar_field("Phi", self_conjugate=False, quantum_numbers={"Q": 1})
psi_q = dirac_field("psi", quantum_numbers={"Q": -1})

mu = S("mu")
m2 = S("m2")

L_u1 = Model(
    (
        m2 * Phi_q.bar * Phi_q
        + Expression.I * psi_q.bar() * Gamma(mu) * CovD(psi_q, mu)
    ),
    gauge_groups=(U1,),
).lagrangian()

delta_u1 = gauge_variation(group=U1, parameter="alpha")
dL_u1 = L_u1.apply_operator(delta_u1)

print("Symbolica view of delta(L):")
print(dL_u1.to_symbolica().expand())

Symbolica view of delta(L):
-g*gamma(bis(4, i_bar_psi_U1),bis(4, i_psi_U1),mink(4, mu))*psi(i_psi_U1)*psibar(i_bar_psi_U1)*PartialD(alpha,mu)+g*gamma(bis(4, i_bar_psi_covd),bis(4, i_psi_covd),mink(4, mu))*psi(i_psi_covd)*psibar(i_bar_psi_covd)*PartialD(alpha,mu)


The raw expression has spinor dummies introduced by the CovD lowering (`i_bar_psi_U1`, `i_psi_covd`, ...). The vertex canonicaliser used in the rest of the pipeline reduces this to **`0`**:

In [47]:
spinor_dummies = (
    S("i_bar_psi_U1"), S("i_psi_U1"),
    S("i_bar_psi_covd"), S("i_psi_covd"),
)
canonical = canonize_full(
    dL_u1.to_symbolica(),
    lorentz_indices=(mu,),
    spinor_indices=spinor_dummies,
)
print("delta(L) canonicalised:")
print(canonical)
assert canonical.to_canonical_string() == "0"
print("\n=> L = m^2 Phibar Phi + i Psibar gamma^mu D_mu Psi is gauge invariant.")

delta(L) canonicalised:
0

=> L = m^2 Phibar Phi + i Psibar gamma^mu D_mu Psi is gauge invariant.


### 11.1 Wrong-charge sanity check

If we couple a charged scalar to a neutral Yukawa pair (so total $U(1)$ charge is non-zero), the variation should **not** vanish — it pinpoints exactly the charge-conservation violation.

In [48]:
psi_neutral = dirac_field("eta", quantum_numbers={"Q": 0})
L_wrong = Model(S("y") * psi_neutral.bar() * psi_neutral() * Phi_q).lagrangian()

dL_wrong = L_wrong.apply_operator(delta_u1)
print("delta(y * eta.bar * eta * Phi)  (Phi has Q=1, eta has Q=0):")
print(dL_wrong.to_symbolica().expand())
print("\n=> non-zero: charge is not conserved by this Yukawa.")

delta(y * eta.bar * eta * Phi)  (Phi has Q=1, eta has Q=0):
1𝑖*Phi*g*alpha*y*eta(i_decl_1)*etabar(i_decl_1)

=> non-zero: charge is not conserved by this Yukawa.


In [49]:
L_charged = Model(S("y") * psi_q.bar() * psi_q() * Phi).lagrangian()
dL_charged = L_charged.apply_operator(delta_u1)
print("delta(y * psi.bar * psi * Phi)  (Phi has Q=1, psi has Q=-1):")
print(dL_charged.to_symbolica().expand())
print("\n=> zero: charge is conserved by this Yukawa.")

delta(y * psi.bar * psi * Phi)  (Phi has Q=1, psi has Q=-1):
0

=> zero: charge is conserved by this Yukawa.


### 11.2 Non-abelian gauge-boson variation

For a non-abelian group the gauge-boson variation has two pieces: an **inhomogeneous** $+\partial_\mu \alpha^a$ and a **homogeneous** $-g f^{abc}\alpha^b A^c_\mu$. We can apply `gauge_variation` to a single $A^a_\mu$ slot to see them produced explicitly.

In [50]:
from model.metadata import COLOR_ADJ_INDEX, COLOR_FUND_INDEX
from model import GaugeRepresentation
from symbolic.spenso_structures import gauge_generator, structure_constant

g_s = S("gS")
G_field =Field(
    "G",
    spin=1,
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
    self_conjugate=True,
)
SU3 = GaugeGroup(
    name="SU3",
    abelian=False,
    coupling=g_s,
    gauge_boson=G_field,
    structure_constant=structure_constant,
    representations=(
        GaugeRepresentation(
            index=COLOR_FUND_INDEX,
            generator_builder=gauge_generator,
            name="fundamental",
        ),
    ),
)

mu_na = S("mu")
aa = S("aa")
L_A = Model(
    G_field(mu_na, aa),
    gauge_groups=(SU3,),
).lagrangian()

delta_su3 = gauge_variation(group=SU3, parameter="alpha")
dL_A = L_A.apply_operator(delta_su3)
print("delta(A^a_mu):")
print(dL_A.to_symbolica().expand())

delta(A^a_mu):
-gS*f(coad(8, aa),coad(8, a_delta_SU3_1),coad(8, a_delta_SU3_2))*alpha(a_delta_SU3_1)*G(mu,a_delta_SU3_2)+PartialD(alpha(aa),mu)


### 11.3 Summary

`gauge_variation(group=..., parameter=...)` is a thin user wrapper over `FieldOperator`. It encodes the **linearized** infinitesimal gauge transformation as a graded derivation on lowered `InteractionTerm`s, so:

- `apply_operator(delta)` is enough to derive $\delta L$ for matter and gauge boson Lagrangians.
- The result is a plain Symbolica expression that can be canonicalised (with `canonize_full`) to expose vanishing.
- The sign convention follows the codebase (`D_μ = ∂_μ - i g A_μ`); the gauge-boson inhomogeneous part is $+\partial_\mu \alpha^a$.
- A fully reduced check of $\delta(-\tfrac14 F^2) = 0$ requires a more aggressive tensor canonicaliser than the minimal one used in the regression tests; the building blocks ($\delta A$ structure, matter-only / fermion-kinetic cancellations) are nevertheless covered by `tests/test_gauge_variation.py`.